[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/registration/registration.ipynb)

# Point-Cloud Registration

Point-cloud registration finds the rigid motion that carries one set of points onto another when their pairwise correspondences are known, such as two range scans of the same object or a measured part matched against its reference model. Because measurement noise perturbs every point, no single rigid motion lands every source point cleanly on its target, and the registration fits the motion that minimizes their mismatch across the cloud.

A rigid motion is modelled as a unit motor moving points by the sandwich `motor >> point`. Multiplying that sandwich on the right by the motor gives the one-sided residual `target * Motor - Motor * source`, a linear map on motors whose summed squared norm `residual.scalar_norm_squared() + residual.dual().scalar_norm_squared()` forms one symmetric quadratic form over rotation and translation together. Solving `misfit.eigh()` measures candidates against the motor's own metric, which assigns zero norm to pure shifts in flat space and therefore selects the optimal translation for every candidate rotation without centring the clouds or choosing an origin, while the identical code registers points on the three-sphere.

In the matrix notation of the Kabsch algorithm, the two clouds are first shifted to their centroids $\bar{\mathbf x} = \frac{1}{N}\sum_k \mathbf x_k$ and $\bar{\mathbf y} = \frac{1}{N}\sum_k \mathbf y_k$, the cross-covariance $H = \sum_k (\mathbf x_k - \bar{\mathbf x})(\mathbf y_k - \bar{\mathbf y})^{\mathsf T} = U\Sigma V^{\mathsf T}$ is factored by singular value decomposition with the reflection fix $R = V\,\mathrm{diag}(1, 1, \det(VU^{\mathsf T}))\,U^{\mathsf T}$, and the translation $\mathbf t = \bar{\mathbf y} - R\bar{\mathbf x}$ is recovered afterward; in Horn's unit-quaternion formulation, the same centroid subtraction precedes a $4 \times 4$ symmetric eigenproblem for the rotation quaternion.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, Extensor, NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure
from examples.geometry.registration import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga, dtype=np.float64).multivector
Point = ga.gatype.antivector()                      # Point-
# Coordinate planes through the origin; the plane w reads a point's weight.
Upright = ga.gatype.from_blades("x y z")            # Plane+
Bivector = ga.gatype.bivector()                     # Bivector+
Motor = ga.gatype.rotor()                           # Motor+

## Noisy correspondences

An anisotropic Gaussian cloud provides the source points, and applying a known translation and rotation followed by independent small random motors on each point produces the noisy target cloud.

In [ ]:
n_points, cloud_spread, seed = scenarios.n_points, scenarios.cloud_spread, scenarios.seed
shift_vector, turn_vector = scenarios.shift_vector, scenarios.turn_vector
noise_sigma = scenarios.noise_sigma

In [ ]:
def point_cloud(n_points: int, spread: np.ndarray, rng: np.random.Generator) -> Point:
    """Gaussian coordinates scaled by `spread` form an anisotropic cloud of `n_points` unit points (`[n_points] Point-`)."""
    coordinates = rng.normal(size=(n_points, ga.dimension - 1)) * spread       # [n_points, 3]
    return (mv(Upright, coordinates) + mv.w).dual().normalized()               # [n_points] Point-


def noisy_cloud(points: Point, sigma: float, rng: np.random.Generator) -> Point:
    """Independent Gaussian bivector generators of scale `sigma / 2` perturb each point (`[...] Point-`)."""
    n_planes = len(Bivector.output_subspace)
    generators = mv.bivector(rng.normal(scale=sigma / 2, size=(*points.shape, n_planes)))  # [...] Bivector+
    perturbations = generators.exp()                                           # [...] Motor+
    return perturbations >> points                                             # [...] Point-


rng = np.random.default_rng(seed)
source = point_cloud(n_points, cloud_spread, rng)                              # [n_points] Point-
shift_generator = (stack((mv.xw, mv.yw, mv.zw)) * shift_vector).sum(axis=0) / 2  # [] Bivector+
turn_generator = (stack((mv.xy, mv.yz, mv.zx)) * turn_vector).sum(axis=0) / 2    # [] Bivector+
ground_truth = shift_generator.exp() * turn_generator.exp()                    # [] Motor+
moved_source = ground_truth >> source                                          # [n_points] Point-
target = noisy_cloud(moved_source, noise_sigma, rng)                           # [n_points] Point-

## One-sided motor eigenproblem

A motor carrying each source point onto its target satisfies `target * motor == motor * source`. Summing the squared norm of that linear residual and of its complement across every pair measures both Euclidean and ideal planes, so the eigenvector of smallest eigenvalue against the motor metric recovers the full rigid motion in one step.

In [ ]:
def fitted_motor(source: Extensor, target: Extensor) -> Extensor:
    """The one-sided motor eigenproblem recovers the unit motor (`[] Motor+`) aligning `source` onto `target` (`[...] Point-`)."""
    Motor = source.algebra.gatype.rotor()                                      # Motor+
    source, target = source.normalized(), target.normalized()                  # [...] Point-, [...] Point-
    # target * Motor == Motor * source is linear in the motor, giving a linear map on motors per pair.
    residual = target * Motor - Motor * source                                 # [...] Point- <- Motor+

    # Where the metric is degenerate on ideal blades, the bulk norm alone discards those planes;
    # adding the bulk norm of the complement sums the squares of every coefficient.
    bulk = residual.scalar_norm_squared()                                      # [...] Scalar+ <- (Motor+, Motor+)
    weight = residual.dual().scalar_norm_squared()                             # [...] Scalar+ <- (Motor+, Motor+)
    misfit = (bulk + weight).sum(axis=0)                                       # [] Scalar+ <- (Motor+, Motor+)
    # Against the motor's own metric, which in flat space has no unit for a shift, eigh picks for each turn
    # the shift that fits best, centring the clouds without choosing an origin.
    values, motors = misfit.eigh()                                             # [8] Scalar+, [8] Motor+
    return motors[values.argmin()].normalized()                                # [] Motor+


alignment = fitted_motor(source, target)                                       # [] Motor+
aligned = alignment >> source                                                  # [n_points] Point-
save_figure(render.registration_figure(source, target, aligned), "registration_one_sided_residual");

Grey balls mark the unaligned source cloud, translucent blue shells surround the noisy target points, and red balls show the aligned source cloud linked to each corresponding target by a short residual rod.

## Registration on the three-sphere

Replacing the degenerate weight plane of flat projective space with a positive-square fourth plane places points and motors on the three-sphere, where rotations in all six coordinate planes have positive norm and the identical `fitted_motor` aligns spherical correspondences without modification.

In [ ]:
spherical_ga = Algebra("w+x+y+z+")
spherical_mv = NumpyContext(spherical_ga, dtype=np.float64).multivector
SphericalUpright = spherical_ga.gatype.from_blades("x y z")                    # Plane+
SphericalBivector = spherical_ga.gatype.bivector()                             # Bivector+

spherical_rng = np.random.default_rng(seed)
spherical_coords = spherical_rng.normal(size=(n_points, 3)) * cloud_spread     # [n_points, 3]
sphere_source = (spherical_mv(SphericalUpright, spherical_coords) + spherical_mv.w).dual().normalized()  # [n_points] Point-
sphere_shift = (stack((spherical_mv.xw, spherical_mv.yw, spherical_mv.zw)) * shift_vector).sum(axis=0) / 2  # [] Bivector+
sphere_turn = (stack((spherical_mv.xy, spherical_mv.yz, spherical_mv.zx)) * turn_vector).sum(axis=0) / 2    # [] Bivector+
sphere_truth = sphere_shift.exp() * sphere_turn.exp()                          # [] Motor+
sphere_moved = sphere_truth >> sphere_source                                   # [n_points] Point-
n_sphere_planes = len(SphericalBivector.output_subspace)
sphere_noise = spherical_mv.bivector(spherical_rng.normal(scale=noise_sigma / 2, size=(n_points, n_sphere_planes)))  # [n_points] Bivector+
sphere_perturbations = sphere_noise.exp()                                      # [n_points] Motor+
sphere_target = sphere_perturbations >> sphere_moved                           # [n_points] Point-

sphere_alignment = fitted_motor(sphere_source, sphere_target)                  # [] Motor+
sphere_aligned = sphere_alignment >> sphere_source                             # [n_points] Point-
save_figure(render.registration_figure(sphere_source, sphere_target, sphere_aligned), "registration_three_sphere");

Projected orthographically onto the equatorial three-space, the grey source cloud on the three-sphere lands inside the translucent blue target shells once carried by the fitted four-dimensional rotor.

In [ ]:
# checks
# Both fitted motors have unit reverse norm and bring the clouds within the perturbation scale of the targets.
assert ((alignment * alignment.reverse() - mv.rotor()).scalar_norm_squared() < 1e-12).all()
assert ((sphere_alignment * sphere_alignment.reverse() - spherical_mv.rotor()).scalar_norm_squared() < 1e-12).all()
euclidean_rms = ((aligned.normalized() - target.normalized()).dual().norm_squared().sum(axis=0) / n_points).square_root()
spherical_rms = ((sphere_aligned.normalized() - sphere_target.normalized()).dual().norm_squared().sum(axis=0) / n_points).square_root()
assert euclidean_rms < 0.15
assert spherical_rms < 0.15